In [0]:
%python
dbutils.widgets.text("catalog", "tpch_sales")
catalog = dbutils.widgets.get("catalog")

# Accumulator for all check results
checks = []

def add_check(category, check_name, status, details, severity="critical"):
    """Record the outcome of one data-quality check."""
    checks.append({
        "category": category,
        "check_name": check_name,
        "status": status,          # PASS / FAIL
        "details": details,
        "severity": severity,      # critical (fails pipeline) / informational
    })

In [0]:
%python
# CHECK 1: All eight source tables were ingested into Bronze (exist and non-empty)
source_tables = ["region", "nation", "customer", "supplier", "part", "orders", "partsupp", "lineitem"]
bronze_counts = {}  # cached for reuse in the rejected-records check

for t in source_tables:
    try:
        rows = spark.table(f"{catalog}.bronze.{t}").count()
        bronze_counts[t] = rows
        status = "PASS" if rows > 0 else "FAIL"
        details = f"rows={rows}"
    except Exception as e:
        bronze_counts[t] = None
        status, details = "FAIL", f"table missing or unreadable ({type(e).__name__})"
    add_check("ingestion", f"bronze.{t} ingested", status, details)

# The Gold layer must also be present before DQ can run
for a in ["dim_geography", "dim_customer", "dim_supplier", "dim_product", "fact_sales", "vw_annual_sales_kpi"]:
    exists = spark.catalog.tableExists(f"{catalog}.gold.{a}")
    add_check("ingestion", f"gold.{a} present", "PASS" if exists else "FAIL",
              "present" if exists else "missing")

print(f"Ingestion checks complete ({len(checks)} checks so far)")

In [0]:
%python
# CHECK 2: Primary keys are populated (no nulls) and unique across Silver and Gold
pk_specs = {
    f"{catalog}.silver.region": "regionkey",
    f"{catalog}.silver.nation": "nationkey",
    f"{catalog}.silver.customer": "custkey",
    f"{catalog}.silver.supplier": "suppkey",
    f"{catalog}.silver.part": "partkey",
    f"{catalog}.silver.orders": "orderkey",
    f"{catalog}.silver.partsupp": "partkey, suppkey",
    f"{catalog}.silver.lineitem": "orderkey, linenumber",
    f"{catalog}.gold.dim_geography": "nation_key",
    f"{catalog}.gold.dim_customer":  "customer_key",
    f"{catalog}.gold.dim_supplier": "supplier_key",
    f"{catalog}.gold.dim_product": "product_key",
    f"{catalog}.gold.fact_sales": "order_key, line_number",
}

for tbl, keys in pk_specs.items():
    key_list = [k.strip() for k in keys.split(",")]
    q = f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT {', '.join(key_list)}) AS distinct_keys,
            SUM(CASE WHEN {' OR '.join(f'{k} IS NULL' for k in key_list)} THEN 1 ELSE 0 END) AS null_keys
        FROM {tbl}
    """
    r = spark.sql(q).first()
    nulls = r["null_keys"] or 0
    ok = (nulls == 0) and (r["total_rows"] == r["distinct_keys"])
    add_check("primary_keys", f"{tbl} ({keys})",
              "PASS" if ok else "FAIL",
              f"rows={r['total_rows']}, distinct={r['distinct_keys']}, nulls={nulls}")

print(f"Primary key checks complete ({len(checks)} checks so far)")

In [0]:
%python
# CHECK 3: Fact table contains no null surrogate (or composite) keys
q = f"""
    SELECT
        SUM(CASE WHEN order_key    IS NULL THEN 1 ELSE 0 END) AS order_key,
        SUM(CASE WHEN line_number  IS NULL THEN 1 ELSE 0 END) AS line_number,
        SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS customer_key,
        SUM(CASE WHEN supplier_key IS NULL THEN 1 ELSE 0 END) AS supplier_key,
        SUM(CASE WHEN product_key  IS NULL THEN 1 ELSE 0 END) AS product_key
    FROM {catalog}.gold.fact_sales
"""
r = spark.sql(q).first()

for col in ["order_key", "line_number", "customer_key", "supplier_key", "product_key"]:
    nulls = r[col] or 0
    add_check("null_surrogate_keys", f"fact_sales.{col} IS NOT NULL",
              "PASS" if nulls == 0 else "FAIL", f"null_rows={nulls}")

print(f"Surrogate key checks complete ({len(checks)} checks so far)")

In [0]:
%python
# CHECK 4: Numeric measures satisfy the documented business rules
# fact_sales measures
q = f"""
    SELECT
        SUM(CASE WHEN quantity <= 0   THEN 1 ELSE 0 END) AS bad_quantity,
        SUM(CASE WHEN extended_price <  0   THEN 1 ELSE 0 END) AS bad_extended_price,
        SUM(CASE WHEN discount <  0 OR discount >= 1 THEN 1 ELSE 0 END) AS bad_discount,
        SUM(CASE WHEN tax <  0 OR tax >= 1 THEN 1 ELSE 0 END) AS bad_tax,
        SUM(CASE WHEN order_total_price <= 0   THEN 1 ELSE 0 END) AS bad_order_total_price
    FROM {catalog}.gold.fact_sales
"""
r = spark.sql(q).first()
fact_rules = {
    "bad_quantity":            "quantity > 0",
    "bad_extended_price":      "extended_price >= 0",
    "bad_discount":            "0 <= discount < 1",
    "bad_tax":                 "0 <= tax < 1",
    "bad_order_total_price":   "order_total_price > 0",
}
for k, rule in fact_rules.items():
    violations = r[k] or 0
    add_check("business_rules", f"fact_sales: {rule}",
              "PASS" if violations == 0 else "FAIL", f"violations={violations}")

# dim_product: retail_price must be non-negative
r = spark.sql(f"""
    SELECT SUM(CASE WHEN retail_price < 0 THEN 1 ELSE 0 END) AS bad
    FROM {catalog}.gold.dim_product
""").first()
add_check("business_rules", "dim_product: retail_price >= 0",
          "PASS" if (r["bad"] or 0) == 0 else "FAIL", f"violations={r['bad'] or 0}")

# silver.partsupp: available quantity and supply cost must be non-negative
r = spark.sql(f"""
    SELECT
        SUM(CASE WHEN availqty   < 0 THEN 1 ELSE 0 END) AS bad_availqty,
        SUM(CASE WHEN supplycost < 0 THEN 1 ELSE 0 END) AS bad_supplycost
    FROM {catalog}.silver.partsupp
""").first()
add_check("business_rules", "partsupp: availqty >= 0",
          "PASS" if (r["bad_availqty"] or 0) == 0 else "FAIL", f"violations={r['bad_availqty'] or 0}")
add_check("business_rules", "partsupp: supplycost >= 0",
          "PASS" if (r["bad_supplycost"] or 0) == 0 else "FAIL", f"violations={r['bad_supplycost'] or 0}")

print(f"Business rule checks complete ({len(checks)} checks so far)")

In [0]:
%python
# CHECK 5: Gold revenue and row counts reconcile with Silver
# Silver side uses the same join as the Gold build; every Silver line item must appear in Gold
q = f"""
    WITH gold_metrics AS (
        SELECT COUNT(*) AS row_count,
               SUM(extended_price * (1 - discount)) AS net_revenue
        FROM {catalog}.gold.fact_sales
    ),
    silver_metrics AS (
        SELECT COUNT(*) AS row_count,
               SUM(l.extendedprice * (1 - l.discount)) AS net_revenue
        FROM {catalog}.silver.lineitem l
        JOIN {catalog}.silver.orders o   ON l.orderkey = o.orderkey
        JOIN {catalog}.silver.customer c ON o.custkey = c.custkey
        JOIN {catalog}.silver.supplier s ON l.suppkey = s.suppkey
        JOIN {catalog}.silver.part p     ON l.partkey = p.partkey
    )
    SELECT g.row_count AS gold_rows, s.row_count AS silver_rows,
           g.net_revenue AS gold_revenue, s.net_revenue AS silver_revenue
    FROM gold_metrics g CROSS JOIN silver_metrics s
"""
r = spark.sql(q).first()

row_diff = r["gold_rows"] - r["silver_rows"]
rev_diff = (r["gold_revenue"] or 0) - (r["silver_revenue"] or 0)

add_check("reconciliation", "Gold vs Silver row count",
          "PASS" if row_diff == 0 else "FAIL",
          f"gold={r['gold_rows']}, silver={r['silver_rows']}, diff={row_diff}")
add_check("reconciliation", "Gold vs Silver net revenue",
          "PASS" if abs(rev_diff) < 0.01 else "FAIL",
          f"gold={round(r['gold_revenue'] or 0, 2)}, silver={round(r['silver_revenue'] or 0, 2)}, diff={round(rev_diff, 2)}")

print(f"Reconciliation checks complete ({len(checks)} checks so far)")

In [0]:
%python
# CHECK 6: Rejected-record (quarantine) counts reported, and Bronze = Silver valid + quarantined
rejected_total = 0
for t in source_tables:
    silver_rows = spark.table(f"{catalog}.silver.{t}").count()
    quarantine_rows = spark.table(f"{catalog}.silver.quarantine_{t}").count()
    rejected_total += quarantine_rows
    bronze_rows = bronze_counts[t]
    balanced = (bronze_rows is not None) and (bronze_rows == silver_rows + quarantine_rows)
    # A non-zero quarantine count is expected behaviour (bad rows are isolated, not lost),
    # so the count is reported informationally; only a balance break is critical.
    add_check("rejected_records", f"silver.{t} balance",
              "PASS" if balanced else "FAIL",
              f"bronze={bronze_rows}, valid={silver_rows}, rejected={quarantine_rows}",
              severity="critical" if not balanced else "informational")

add_check("rejected_records", "total rejected records across all tables",
          "PASS", f"total_quarantined_rows={rejected_total}", severity="informational")

print(f"Rejected-record checks complete ({len(checks)} checks so far). Total quarantined rows: {rejected_total}")

In [0]:
%python
from pyspark.sql import Row

# Assemble the report, display it, and persist it for auditing
results_df = spark.createDataFrame([Row(**c) for c in checks])
display(results_df)

results_df.write.mode("overwrite").saveAsTable(f"{catalog}.gold.dq_results")
print(f"Report written to {catalog}.gold.dq_results")

# FAIL GATE: any critical violation raises and fails the pipeline run
failures = [c for c in checks if c["status"] == "FAIL"]
if failures:
    msg = "\n".join(f"  [{c['category']}] {c['check_name']} -> {c['details']}" for c in failures)
    raise RuntimeError(f"DATA QUALITY GATE FAILED - {len(failures)} critical violation(s):\n{msg}")

print(f"ALL {len(checks)} DATA QUALITY CHECKS PASSED - pipeline may proceed")